# Sentiment Scoring Only (Qwen fine-tuned adapter)

Runs the expensive/GPU-bound part of the pipeline exactly once: streams FNSPID from
Hugging Face (no full download), classifies every matched headline with the fine-tuned
Qwen2.5-3B-Instruct + LoRA adapter, and saves the headline-level scores to `scored_news.csv`.

This notebook does **not** touch stock prices, technical indicators, or RL training --
those stay local (fast, no GPU needed). Download `scored_news.csv` from Kaggle's Output
tab when this finishes and bring it back for the merge + training step.


### Install packages

In [1]:
!pip install --upgrade pip -q

# HuggingFace stack (Kaggle provides CUDA already)
!pip install "transformers>=4.46" tokenizers huggingface-hub datasets peft accelerate -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.7 MB/s eta 0:00:0000:0100:01


### Stream FNSPID (no full dataset download -- only matching rows kept)

In [2]:
from datasets import load_dataset
import pandas as pd

TOP10 = set(['AAPL','MSFT','AMZN','GOOGL','META',
             'NVDA','JPM','TSLA','BA','DIS'])
DATE_MIN = pd.Timestamp("2020-01-01")
DATE_MAX = pd.Timestamp("2023-12-31 23:59:59")

# FNSPID has a single "train" split -- no train/test split on the HF side.
ds = load_dataset("Zihan1004/FNSPID", split="train", streaming=True)
# Drop the heavy text/summary columns we don't use -- cuts bandwidth substantially since the
# underlying parquet is columnar. This dataset's repo metadata is incomplete, so
# ds.column_names can come back None instead of a list -- fall back to a no-op in that case
# (correctness doesn't depend on this, it's purely a bandwidth optimization).
_known_cols = ds.column_names or []
_drop_cols = [c for c in ["Url", "Publisher", "Author", "Article",
                           "Lsa_summary", "Luhn_summary",
                           "Textrank_summary", "Lexrank_summary"]
              if c in _known_cols]
if _drop_cols:
    ds = ds.remove_columns(_drop_cols)

keep_rows = []
batch = {"Date": [], "Article_title": [], "Stock_symbol": []}
BATCH_SIZE = 10_000
n_seen = 0

def _flush(batch, keep_rows):
    if not batch["Date"]:
        return
    chunk = pd.DataFrame(batch)
    chunk["Date_parsed"] = pd.to_datetime(chunk["Date"], errors="coerce", utc=True).dt.tz_localize(None)
    mask = (chunk["Date_parsed"] >= DATE_MIN) & (chunk["Date_parsed"] <= DATE_MAX) & chunk["Stock_symbol"].isin(TOP10)
    sub = chunk.loc[mask, ["Date", "Article_title", "Stock_symbol"]]
    if not sub.empty:
        keep_rows.append(sub)

for row in ds:
    n_seen += 1
    batch["Date"].append(row["Date"])
    batch["Article_title"].append(row["Article_title"])
    batch["Stock_symbol"].append(row["Stock_symbol"])
    if len(batch["Date"]) >= BATCH_SIZE:
        _flush(batch, keep_rows)
        batch = {"Date": [], "Article_title": [], "Stock_symbol": []}
        if n_seen % 500_000 == 0:
            kept_so_far = sum(len(k) for k in keep_rows)
            print(f"streamed {n_seen:,} rows so far, kept {kept_so_far:,}")

_flush(batch, keep_rows)  # last partial batch

df_news = pd.concat(keep_rows, ignore_index=True)
df_news = df_news.rename(columns={
    "Date": "date",
    "Article_title": "headline",
    "Stock_symbol": "ticker",
})
df_news["date"] = pd.to_datetime(df_news["date"], utc=True).dt.tz_localize(None)

print("Final shape:", df_news.shape)
df_news.head()


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


streamed 500,000 rows so far, kept 739
streamed 1,000,000 rows so far, kept 1,572
streamed 1,500,000 rows so far, kept 2,714
streamed 2,000,000 rows so far, kept 2,714
streamed 2,500,000 rows so far, kept 2,746
streamed 3,000,000 rows so far, kept 3,668
streamed 3,500,000 rows so far, kept 4,318
streamed 4,000,000 rows so far, kept 4,318
streamed 4,500,000 rows so far, kept 4,318


/tmp/ipykernel_58/1644155097.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk["Date_parsed"] = pd.to_datetime(chunk["Date"], errors="coerce", utc=True).dt.tz_localize(None)


streamed 5,000,000 rows so far, kept 4,318
streamed 5,500,000 rows so far, kept 4,318
streamed 6,000,000 rows so far, kept 4,318
streamed 6,500,000 rows so far, kept 4,318
streamed 7,000,000 rows so far, kept 4,318
streamed 7,500,000 rows so far, kept 4,318
streamed 8,000,000 rows so far, kept 4,318
streamed 8,500,000 rows so far, kept 4,318
streamed 9,000,000 rows so far, kept 4,318
streamed 9,500,000 rows so far, kept 4,318
streamed 10,000,000 rows so far, kept 4,318
streamed 10,500,000 rows so far, kept 4,318
streamed 11,000,000 rows so far, kept 4,318
streamed 11,500,000 rows so far, kept 4,318
streamed 12,000,000 rows so far, kept 4,318
streamed 12,500,000 rows so far, kept 4,318
streamed 13,000,000 rows so far, kept 4,318
streamed 13,500,000 rows so far, kept 22,000
streamed 14,000,000 rows so far, kept 29,860
streamed 14,500,000 rows so far, kept 29,860
streamed 15,000,000 rows so far, kept 47,313
streamed 15,500,000 rows so far, kept 56,025
streamed 16,000,000 rows so far, kept

KeyError: 'Date'

### Load fine-tuned Qwen adapter and classify every headline

In [3]:
# Add your HF token as a Kaggle secret named HF_TOKEN (Add-ons > Secrets), or set the
# HF_TOKEN environment variable before running -- needed since the adapter repo is private.
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import login
import os

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_REPO = "vijay5799760/Qwen2.5-3B-SEntFiN-LoRA-HPtuned"  # your fine-tuned adapter (private repo)

if "HF_TOKEN" in os.environ:
    login(os.environ["HF_TOKEN"])
else:
    try:
        from kaggle_secrets import UserSecretsClient
        login(UserSecretsClient().get_secret("HF_TOKEN"))
    except Exception as e:
        raise RuntimeError(
            "Could not find an HF token. On Kaggle: Add-ons > Secrets > add HF_TOKEN, "
            "enable it for this notebook, then re-run this cell."
        ) from e

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "left"  # last token of every padded sequence is always the true final token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.to(device)
model.eval()

# Each label is verified to be a single token for this tokenizer.
LABEL_IDS = {
    "positive": tokenizer.encode("positive", add_special_tokens=False)[0],
    "negative": tokenizer.encode("negative", add_special_tokens=False)[0],
    "neutral":  tokenizer.encode("neutral",  add_special_tokens=False)[0],
}
LABEL_ID_LIST = [LABEL_IDS["positive"], LABEL_IDS["negative"], LABEL_IDS["neutral"]]

TICKER_TO_COMPANY = {
    "AAPL": "Apple", "MSFT": "Microsoft", "AMZN": "Amazon", "GOOGL": "Alphabet",
    "META": "Meta", "NVDA": "Nvidia", "JPM": "JPMorgan Chase", "TSLA": "Tesla",
    "BA": "Boeing", "DIS": "Disney",
}

SYSTEM_PROMPT = (
    "You are a financial sentiment analyst. Given a news headline and a company name, "
    "classify the sentiment towards that company. "
    "Reply with exactly one word: positive, negative, or neutral."
)

def make_prompt(headline, ticker):
    company = TICKER_TO_COMPANY.get(ticker, ticker)
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\nHeadline: {headline}\nCompany: {company}\nSentiment:<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

@torch.no_grad()
def classify_headlines(texts, tickers, batch_size=32, log_every=500):
    """Returns a continuous score per headline: P(positive) - P(negative), in [-1, 1] --
    read directly from the model's own next-token probability distribution
    (a single forward pass, no generation)."""
    results, t0 = [], time.time()
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        batch_tickers = tickers[i:i + batch_size]
        prompts = [make_prompt(h, t) for h, t in zip(batch_texts, batch_tickers)]
        enc = tokenizer(prompts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        logits = model(**enc).logits[:, -1, :]
        label_logits = logits[:, LABEL_ID_LIST]
        probs = torch.softmax(label_logits, dim=-1)
        scores = (probs[:, 0] - probs[:, 1]).float().cpu().tolist()
        results.extend(scores)
        if i % log_every == 0 and i > 0:
            elapsed = time.time() - t0
            rate = i / elapsed
            eta = (len(texts) - i) / rate
            print(f"{i}/{len(texts)} processed ({elapsed:.1f}s elapsed, ~{eta:.0f}s remaining)")
    return results


RuntimeError: Could not find an HF token. On Kaggle: Add-ons > Secrets > add HF_TOKEN, enable it for this notebook, then re-run this cell.

In [ ]:
df_news["score"] = classify_headlines(df_news["headline"].tolist(), df_news["ticker"].tolist())
df_news.head()


### Save -- download this from Kaggle's Output tab and bring it back locally

In [ ]:
df_news.to_csv("scored_news.csv", index=False)
print("Saved", df_news.shape[0], "scored headlines to scored_news.csv")
df_news["score"].describe()
